# Notebook 2 — Data Exploration & Bronze → Silver

**Purpose:** Explore the raw Bronze data to understand its shape and quality,
then clean and validate it into a Silver Delta table.

In [0]:
# ============================================================
# Defining catalog and schema location in the beginning 
# ============================================================

CATALOG_NAME = "iot_pipeline"
SCHEMA_NAME  = "sensors"

BRONZE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_sensors"
SILVER_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_sensors"

# Valid sensor ranges
# In a real project these would live in a shared config file.
SENSOR_RANGES = {
    "temperature": {"min": 60.0,  "max": 120.0},
    "pressure":    {"min": 1.0,   "max": 10.0},
    "vibration":   {"min": 0.01,  "max": 2.0},
}

print(f"Bronze table : {BRONZE_TABLE}")
print(f"Silver table : {SILVER_TABLE}")

## Part 1 — Data Exploration

In [0]:
df = spark.table(BRONZE_TABLE)

df.printSchema()

In [0]:
row_count = df.count()
col_count = len(df.columns)
print(f"Row count: {row_count}")
print(f"Column count: {col_count}")
print(f"Columns : {df.columns}")

In [0]:
display(df.limit(20))

Initial analysis of the columns:  
- 'event_id' : unique id for each sensor and should not have any duplicates 
- 'value' : actual sensor reading can need to analyze for range and null values 
- 'timestamp' : time at which reading was recorded by the machine 
- 'ingested_at' : time at which the pipeline received the record 

In [0]:
print("Readings per machine")
display(df.groupBy("machine_id").count().orderBy("machine_id"))
print("Readings per sensor type")
display(df.groupBy("sensor_type").count().orderBy("sensor_type"))
display("Readings per machine and sensor type")
display(df.groupBy("machine_id","sensor_type").count().orderBy("machine_id", "sensor_type"))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

#### Time coverage

In [0]:
from pyspark.sql import functions as F

display(
    df.select(
        F.min("timestamp").alias("earliest_reading"),
        F.max("timestamp").alias("latest_reading"),
        F.countDistinct(F.to_date("timestamp")).alias("distinct_days"),
        F.min("ingested_at").alias("first_ingested"),
        F.max("ingested_at").alias("last_ingested"),
    )
)



#### Value Distribution 

In [0]:
print(" Overall value stats ")
display(df.select("value").describe())

print(" Value stats per sensor type")
display(
    df.groupBy("sensor_type")
      .agg(
          F.count("value").alias("count_non_null"),
          F.round(F.mean("value"), 4).alias("mean"),
          F.round(F.stddev("value"), 4).alias("stddev"),
          F.round(F.min("value"), 4).alias("min"),
          F.round(F.max("value"), 4).alias("max"),
      )
      .orderBy("sensor_type")
)

#### Null value check 

In [0]:
total_rows = df.count()

null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).collect()[0].asDict()

print(" Null counts per column ")
for col_name, null_count in null_counts.items():
    pct = null_count / total_rows * 100
    status = "PROBLEM" if null_count > 0 else "OK"
    print(f"  {col_name:<15}: {null_count:>5} nulls ({pct:.1f}%)  [{status}]")

### Duplicate event ID check 

In [0]:
duplicate_ids = (
    df.groupBy("event_id")
      .count()
      .filter(F.col("count") > 1)
)

duplicate_count = duplicate_ids.count()
duplicate_rows  = df.count() - df.dropDuplicates(["event_id"]).count()

print(f"Distinct event_ids with duplicates : {duplicate_count:,}")
print(f"Total extra rows to remove         : {duplicate_rows:,}")
print(f"Duplicate rate                     : {duplicate_rows/total_rows*100:.1f}%")

print("\n Sample duplicate event IDs ")
display(duplicate_ids.orderBy(F.col("count").desc()).limit(10))

#### Value range check 

In [0]:
from functools import reduce

# A flag column: True if value is outside valid range for its sensor type
range_condition = reduce(
    lambda a, b: a | b,
    [
        (F.col("sensor_type") == stype) &
        (
            (F.col("value") < cfg["min"]) |
            (F.col("value") > cfg["max"])
        )
        for stype, cfg in SENSOR_RANGES.items()
    ]
)

out_of_range_count = df.filter(range_condition & F.col("value").isNotNull()).count()

print(f"Out-of-range readings : {out_of_range_count:,} ({out_of_range_count/total_rows*100:.1f}%)")

print("\nOut-of-range samples per sensor type ")
display(
    df.filter(range_condition)
      .groupBy("sensor_type")
      .agg(
          F.count("*").alias("out_of_range_count"),
          F.round(F.min("value"), 2).alias("min_bad_value"),
          F.round(F.max("value"), 2).alias("max_bad_value"),
      )
)

In [0]:
# Quality Summary 
print("=" * 50)
print("  DATA QUALITY SUMMARY — BRONZE TABLE")
print("=" * 50)
print(f"  Total rows              : {total_rows:,}")
print(f"  Null values             : {null_counts['value']:,}  ({null_counts['value']/total_rows*100:.1f}%)")
print(f"  Duplicate event IDs     : {duplicate_rows:,}  ({duplicate_rows/total_rows*100:.1f}%)")
print(f"  Out-of-range readings   : {out_of_range_count:,}  ({out_of_range_count/total_rows*100:.1f}%)")
print("=" * 50)
expected_silver = total_rows - null_counts['value'] - duplicate_rows - out_of_range_count
print(f"  Expected Silver rows    : ~{expected_silver:,}")
print("=" * 50)

##  Introduction to Silver transformation

**Rules to apply:**
1. Drop rows where `value` is null
2. Deduplicate on `event_id` — keep the first occurrence of non null value
3. Drop rows where `value` is outside the valid sensor range
4. Add a `quality_checked_at` timestamp — audit trail

In [0]:

df_silver = (
    df
    # Step 1 — remove duplicate event_ids, keep first occurrence
    .dropDuplicates(["event_id"])

    # Step 2 — remove rows where sensor value is null
    .filter(F.col("value").isNotNull())

    # Step 3 — remove out-of-range values using the same
    # range_condition we built during exploration
    .filter(~range_condition)

    # Step 4 — add audit column so we know when this row
    # was cleaned and written to Silver
    .withColumn("quality_checked_at", F.current_timestamp())
)

silver_count = df_silver.count()
removed      = total_rows - silver_count

print(f"Bronze rows : {total_rows:,}")
print(f"Silver rows : {silver_count:,}")
print(f"Removed     : {removed:,}  ({removed/total_rows*100:.1f}% of Bronze)")

In [0]:
print("Silver sample")
display(df_silver.limit(20))

print("Silver value stats — should be within valid ranges ")
display(
    df_silver.groupBy("sensor_type")
      .agg(
          F.count("*").alias("row_count"),
          F.round(F.min("value"), 4).alias("min_value"),
          F.round(F.max("value"), 4).alias("max_value"),
          F.round(F.mean("value"), 4).alias("mean_value"),
      )
      .orderBy("sensor_type")
)

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("sensor_type", "machine_id")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table saved: {SILVER_TABLE}")

In [0]:
df_verify = spark.table(SILVER_TABLE)

print("=== Silver table schema ===")
df_verify.printSchema()

print("\n=== Row count ===")
print(f"Silver rows: {df_verify.count():,}")

print("\n=== Null check — all should be 0 ===")
null_check = df_verify.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_verify.columns
]).collect()[0].asDict()

for col_name, null_count in null_check.items():
    status = "OK" if null_count == 0 else "STILL HAS NULLS — check your logic"
    print(f"  {col_name:<20}: {null_count} nulls  [{status}]")

print("\n=== Duplicate check — should be 0 ===")
dup_check = df_verify.count() - df_verify.dropDuplicates(["event_id"]).count()
print(f"  Duplicate event_ids: {dup_check}  [{'OK' if dup_check == 0 else 'STILL HAS DUPLICATES'}]")